In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# —— 1. 读取数据 ——  
# —— 1. Load data ——  
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df  = pd.read_csv('/kaggle/input/titanic/test.csv')

# —— 2. 特征工程函数 ——  
# —— 2. Feature engineering function ——  
# —— 特征工程：新增 Title, CabinLetter, FamilySize, TicketPrefix, FareBin ——  
# —— Feature Engineering: extract Title, CabinLetter, FamilySize, TicketPrefix, FareBin ——  
import pandas as pd

def add_features(df):
    # a) 从 Name 提取称谓 Title  
    # a) Extract Title from Name
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)

    # b) Cabin 首字母，缺失填 'U'  
    # b) Cabin letter (first char), fill missing with 'U'
    df['CabinLetter'] = df['Cabin'].fillna('U').str[0]

    # c) 家庭规模 FamilySize = SibSp + Parch + 1  
    # c) FamilySize = SibSp + Parch + 1
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # d) Ticket 前缀 TicketPrefix  
    # d) Ticket prefix
    df['TicketPrefix'] = df['Ticket']\
        .str.extract(r'^([A-Za-z\.\/]+)', expand=False)\
        .fillna('NONE')

    # e) Fare 分位分箱 FareBin  
    # e) Fare binning by quantile
    df['FareBin'] = pd.qcut(df['Fare'].fillna(-1), 4, labels=False)

    return df

# 应用到训练和测试数据  
# Apply to train and test
train_df = add_features(train_df)
test_df  = add_features(test_df)


# 应用特征工程  
# Apply feature engineering
train_df = add_features(train_df)
test_df  = add_features(test_df)

# —— 3. 准备特征矩阵与标签 ——  
# —— 3. Prepare features and target ——  
X = train_df.drop(columns=['Survived','PassengerId','Name','Ticket','Cabin'])
y = train_df['Survived']

# 数值与类别特征列  
# Numeric and categorical columns  
# num_cols = ['Age','Fare','FamilySize','SibSp','Parch']
# cat_cols = ['Pclass','Sex','Embarked','Title','CabinLetter']
# —— 更新特征列列表 ——  
# —— Update feature lists ——  
num_cols = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch', 'FareBin']         # + FareBin
cat_cols = ['Pclass', 'Sex', 'Embarked', 'Title', 'CabinLetter', 'TicketPrefix']  # + TicketPrefix


# —— 4. 构建预处理流水线 ——  
# —— 4. Build preprocessing pipelines ——  
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # 中位数填充／median imputation
    ('scaler', StandardScaler())                     # 标准化／standard scaling
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),  # 常数填充
    ('onehot', OneHotEncoder(handle_unknown='ignore'))                      # One‑Hot 编码
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# —— 5. 定义基模型 & Stacking ——  
# —— 5. Define base learners & Stacking ——  
estimators = [
    ('rf',  RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                          n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', stacking_clf)
])

# —— 6. 分层交叉验证评估 ——  
# —— 6. Stratified K‑Fold CV evaluation ——  
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print("各折得分／Fold scores:", np.round(scores, 4))
print("平均准确率／Mean Accuracy: ", np.round(scores.mean(), 4))


各折得分／Fold scores: [0.8547 0.8596 0.809  0.8483 0.8427]
平均准确率／Mean Accuracy:  0.8429


In [3]:
# —— Optuna 自动化超参调优 ——  
# —— Optuna hyperparameter tuning ——  
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier

def objective(trial):
    # 采样超参数 Sample hyperparameters  
    rf_n   = trial.suggest_int('rf_n_estimators', 100, 500)
    rf_d   = trial.suggest_int('rf_max_depth',      4,   12)
    xgb_n  = trial.suggest_int('xgb_n_estimators', 100, 500)
    xgb_d  = trial.suggest_int('xgb_max_depth',     3,   10)
    xgb_lr = trial.suggest_float('xgb_lr', 0.01, 0.3)

    # 构建基模型 Re‑build base learners  
    estimators = [
        ('rf',  RandomForestClassifier(n_estimators=rf_n, max_depth=rf_d, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                              n_estimators=xgb_n, max_depth=xgb_d,
                              learning_rate=xgb_lr, random_state=42))
    ]
    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000),
        cv=StratifiedKFold(5, shuffle=True, random_state=42)
    )
    pipeline.set_params(clf=stack)
    score = cross_val_score(
        pipeline, X, y,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='accuracy', n_jobs=-1
    ).mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("最佳超参／Best params:", study.best_trial.params)
print("最好 CV 分数／Best CV score:", study.best_value)


[I 2025-04-22 10:05:07,442] A new study created in memory with name: no-name-466e6f43-4c79-4905-8288-3cc12e7cbfb7
[I 2025-04-22 10:05:22,355] Trial 0 finished with value: 0.8394890465130876 and parameters: {'rf_n_estimators': 469, 'rf_max_depth': 6, 'xgb_n_estimators': 262, 'xgb_max_depth': 5, 'xgb_lr': 0.29617798270106516}. Best is trial 0 with value: 0.8394890465130876.
[I 2025-04-22 10:05:41,193] Trial 1 finished with value: 0.8305191136777352 and parameters: {'rf_n_estimators': 450, 'rf_max_depth': 12, 'xgb_n_estimators': 290, 'xgb_max_depth': 6, 'xgb_lr': 0.16467725242103073}. Best is trial 0 with value: 0.8394890465130876.
[I 2025-04-22 10:05:53,361] Trial 2 finished with value: 0.8383717280773334 and parameters: {'rf_n_estimators': 293, 'rf_max_depth': 4, 'xgb_n_estimators': 484, 'xgb_max_depth': 10, 'xgb_lr': 0.1674892432411616}. Best is trial 0 with value: 0.8394890465130876.
[I 2025-04-22 10:06:06,535] Trial 3 finished with value: 0.8282719226664993 and parameters: {'rf_n_est

最佳超参／Best params: {'rf_n_estimators': 203, 'rf_max_depth': 8, 'xgb_n_estimators': 421, 'xgb_max_depth': 3, 'xgb_lr': 0.19373685475319002}
最好 CV 分数／Best CV score: 0.8473542150524136


In [4]:
# —— 7. 在全量训练集上训练模型 & 生成提交文件 ——  
# —— 7. Fit on full train set & create submission.csv ——  

# 先用全量训练集训练 Pipeline  
# Fit the pipeline on the entire training data
pipeline.fit(X, y)

# 准备测试集特征矩阵（与训练时 drop 同样的列）  
# Prepare test features (drop same columns as train)
X_test = test_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# 对测试集做预测  
# Predict on test set
test_preds = pipeline.predict(X_test)

# 构造提交 DataFrame  
# Build submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_preds
})

# 保存为 submission.csv  
# Save to submission.csv
submission.to_csv('submission.csv', index=False)

print("Submission file created: submission.csv")


Submission file created: submission.csv
